# Exploring HWP vs rotations...

In [ ]:
# Third-party imports
import attrs
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import os

from frozendict import frozendict

IS_FAST_MODE = os.environ.get("FLASQ_FAST_MODE_OVERRIDE", "False").lower() == "true"

# Local application imports
from qualtran.surface_code.flasq.flasq_model import (
    conservative_FLASQ_costs,
    optimistic_FLASQ_costs,
)
from qualtran.surface_code.flasq.optimization import (
    run_sweep,
    post_process_for_logical_depth,
    generate_circuit_specific_configs,
)
from qualtran.surface_code.flasq.examples.hwp import (
    build_hwp_circuit,
    build_parallel_rz_circuit,
)
from qualtran.surface_code.flasq.naive_grid_qubit_manager import NaiveGridQubitManager


sns.set_context(context="paper", font_scale=1.1)

In [ ]:
# --- Global Parameters ---
# This cell contains all the global parameters for the notebook.

# --- Error Budget Parameters ---
# We define the error budget per qubit we are trying to rotate.
# This allows for a fair comparison of subroutines that act on different numbers of qubits.
target_rotation_error_per_qubit_to_rotate = 1e-7  # e.g., 3e-3 / 30000
target_cultivation_error_per_qubit_to_rotate = 1e-5  # e.g., 3e-1 / 30000

# --- Physical & QEC Parameters ---
physical_error_rate = (
    1e-3  # Assumed physical error rate for deriving cultivation costs.
)
reference_code_distance = 14  # Reference code distance for deriving a realistic vcult_factor.
reaction_time_in_cycles = 0.0  # Latency for classical control. Set to 0 because the HWP implementation is linear depth (whereas it could be made log depth easily.)

In [ ]:
# --- Define Circuit Builder Wrappers for Sweeping ---
# This cell contains the functions that build the circuits for the sweep.
# By keeping them in a separate cell, we can avoid re-defining them
# when we only want to change the sweep parameters in the cells below.


def build_hwp_circuit_for_sweep(n_qubits_data, angle):
    """Wrapper for HWP builder to support the sweep engine.

    This function returns the circuit and conversion kwargs as a tuple, which
    `analyze_logical_circuit` can handle. The check for sufficient qubits is
    handled later in the pipeline by `calculate_single_flasq_summary`.
    """

    data_qubit_manager = NaiveGridQubitManager(max_cols=10, negative=False)
    ancilla_qubit_manager = NaiveGridQubitManager(max_cols=10, negative=True)

    hwp_bloq, hwp_circuit, hwp_data_qubits = build_hwp_circuit(
        n_qubits_data=n_qubits_data,
        angle=angle,
        data_qubit_manager=data_qubit_manager,
        ancilla_qubit_manager=ancilla_qubit_manager,
    )

    in_quregs = frozendict({"x": tuple(hwp_data_qubits)})
    flasq_conversion_kwargs = {
        "signature": hwp_bloq.signature,
        "qubit_manager": ancilla_qubit_manager,
        "in_quregs": in_quregs,
        "out_quregs": in_quregs,
    }
    return hwp_circuit, flasq_conversion_kwargs


# The default Rz builder returns (circuit, qubits). We need just the circuit.
build_parallel_rz_for_sweep = lambda **kwargs: build_parallel_rz_circuit(**kwargs)[0]

# Example: Logical Cost Comparison

This section is preserved from the original notebook. It provides a useful, step-by-step walkthrough of the cost analysis for a single data point, which is good for pedagogical purposes. The subsequent sections will use the new, automated sweep engine.

In this notebook, we are primarily concerned with logical resource costs, which are independent of the code distance. However, the `V_CULT_FACTOR` used in T-gate costing *does* depend on the code distance. To bridge this, we use a placeholder code distance to derive a realistic `V_CULT_FACTOR` from our cultivation data. This derived factor is then used throughout the notebook.

In [ ]:
# --- Define Parameters for this Example ---
n_qubits_data = 7
angle = 0.123
n_total_logical_qubits = 25  # Total qubit budget for the computation
total_rotation_error = target_rotation_error_per_qubit_to_rotate * n_qubits_data
total_cultivation_error = target_cultivation_error_per_qubit_to_rotate * n_qubits_data

# --- Derive Circuit-Specific Configurations ---
hwp_kwargs = frozendict({"n_qubits_data": n_qubits_data, "angle": angle})
rz_kwargs = frozendict({"n_qubits_data": n_qubits_data, "angle": angle})

hwp_core_config, hwp_total_rot_error = generate_circuit_specific_configs(
    circuit_builder=build_hwp_circuit_for_sweep,
    circuit_builder_kwargs=hwp_kwargs,
    total_synthesis_error=total_rotation_error,
    total_cultivation_error=total_cultivation_error,
    phys_error_rate=physical_error_rate,
    reference_code_distance=reference_code_distance,
)

rz_core_config, rz_total_rot_error = generate_circuit_specific_configs(
    circuit_builder=build_parallel_rz_for_sweep,
    circuit_builder_kwargs=rz_kwargs,
    total_synthesis_error=total_rotation_error,
    total_cultivation_error=total_cultivation_error,
    phys_error_rate=physical_error_rate,
    reference_code_distance=reference_code_distance,
)

# --- Run Minimal Sweeps for this Example ---
phys_qubits_per_logical = 2 * (reference_code_distance + 1) ** 2
n_phys_qubits_total = n_total_logical_qubits * phys_qubits_per_logical

hwp_results = run_sweep(
    circuit_builder_func=build_hwp_circuit_for_sweep,
    circuit_builder_kwargs_list=[hwp_kwargs],
    core_configs_list=[hwp_core_config],
    total_allowable_rotation_error_list=[hwp_total_rot_error],
    reaction_time_in_cycles_list=[reaction_time_in_cycles],
    flasq_model_configs=[(conservative_FLASQ_costs, "Conservative")],
    n_phys_qubits_total_list=[n_phys_qubits_total],
    print_level=0,
)

rz_results = run_sweep(
    circuit_builder_func=build_parallel_rz_for_sweep,
    circuit_builder_kwargs_list=[rz_kwargs],
    core_configs_list=[rz_core_config],
    total_allowable_rotation_error_list=[rz_total_rot_error],
    reaction_time_in_cycles_list=[reaction_time_in_cycles],
    flasq_model_configs=[(conservative_FLASQ_costs, "Conservative")],
    n_phys_qubits_total_list=[n_phys_qubits_total],
    print_level=0,
)

# --- Post-process and extract final summaries ---
hwp_df = post_process_for_logical_depth(hwp_results)
rz_df = post_process_for_logical_depth(rz_results)

hwp_summary_resolved = hwp_df.iloc[0]
parallel_rz_summary_resolved = rz_df.iloc[0]

# --- Print Comparative Summary ---
print("\n" + "=" * 50)
print("--- Example: Logical Cost Comparison ---")
print(
    f"  Input Algorithmic Qubits (HWP): {hwp_summary_resolved['Number of Algorithmic Qubits']}"
)
print(
    f"  Input Algorithmic Qubits (Rz): {parallel_rz_summary_resolved['Number of Algorithmic Qubits']}"
)
print(f"  Total Logical Qubit Budget: {n_total_logical_qubits}")
print(f"  Total Rotation Error Target: {total_rotation_error:.1e}")
print(
    f"  Reference Code Distance: {reference_code_distance} (for vcult derivation)"
)
print("-" * 50)
print("  Derived Cultivation Parameters (HWP):")
print(f"    Target T gate error rate: {hwp_core_config.target_t_error:.1e}")
print(
    f"    Source Cultivation Distance: {hwp_core_config.cultivation_data_source_distance}"
)
print(f"    Derived V_CULT Factor: {hwp_core_config.vcult_factor:.2f}")
print(
    f"    Resulting Cultivation Error Rate: {hwp_core_config.cultivation_error_rate:.2e}"
)
print("\n  Derived Cultivation Parameters (Rz):")
print(f"    Target T gate error rate: {rz_core_config.target_t_error:.1e}")
print(
    f"    Source Cultivation Distance: {rz_core_config.cultivation_data_source_distance}"
)
print(f"    Derived V_CULT Factor: {rz_core_config.vcult_factor:.2f}")
print(
    f"    Resulting Cultivation Error Rate: {rz_core_config.cultivation_error_rate:.2e}"
)
print("-" * 50)

print(f"  FLASQ Model: {hwp_summary_resolved['FLASQ Model']}")
print("=" * 50)

print("\n  Method 1: Hamming Weight Phasing")
print("-" * 35)
print("  Resource Counts:")
print(
    f"    Internal Ancillas: {hwp_summary_resolved['Number of Algorithmic Qubits'] - n_qubits_data}"
)
print(f"    Fluid Ancillas: {hwp_summary_resolved['Number of Fluid Ancilla Qubits']}")
print(f"    Total T-gates: {hwp_summary_resolved['T-Count']:,.0f}")
print(f"    Total Rotations: {hwp_summary_resolved['Total Rotation Count']:,.0f}")
print("  Performance Metrics:")
print(
    f"    Total Logical Depth: {hwp_summary_resolved['Depth (Logical Timesteps)']:,.2f}"
)
print(
    f"    Total Spacetime Volume: {hwp_summary_resolved['Total Spacetime Volume']:,.2f}"
)

print("\n  Method 2: Parallel Rz")
print("-" * 35)
print("  Resource Counts:")
print("    Internal Ancillas: 0")
print(
    f"    Fluid Ancillas: {parallel_rz_summary_resolved['Number of Fluid Ancilla Qubits']}"
)
print(f"    Total T-gates: {parallel_rz_summary_resolved['T-Count']:,.0f}")
print(
    f"    Total Rotations: {parallel_rz_summary_resolved['Total Rotation Count']:,.0f}"
)
print("  Performance Metrics:")
print(
    f"    Total Logical Depth: {parallel_rz_summary_resolved['Depth (Logical Timesteps)']:,.2f}"
)
print(
    f"    Total Spacetime Volume: {parallel_rz_summary_resolved['Total Spacetime Volume']:,.2f}"
)
print("=" * 50)

# Automated Sweep using the new `run_sweep` Engine

In [ ]:
# --- Define Sweep Parameters ---
if IS_FAST_MODE:
    n_qubits_data_list = [15]
else:
    n_qubits_data_list = [15, 43, 121]

# --- Fixed Parameters for Sweep ---
angle = 0.1

# Full Span (0% discount) - Already defined globally

# Zero Span (100% discount)
conservative_zero_span = attrs.evolve(
    conservative_FLASQ_costs, connect_span_volume=0.0, compute_span_volume=0.0
)
optimistic_zero_span = attrs.evolve(
    optimistic_FLASQ_costs, connect_span_volume=0.0, compute_span_volume=0.0
)

# Update flasq_model_configs to define the bounds, removing 'Half Span'.
# Use descriptive names for post-processing.
flasq_model_configs = [
    (conservative_FLASQ_costs, "Conservative (Full Span)"),
    (conservative_zero_span, "Conservative (Zero Span)"),
    (optimistic_FLASQ_costs, "Optimistic (Full Span)"),
    (optimistic_zero_span, "Optimistic (Zero Span)"),
]
if IS_FAST_MODE:
    n_total_logical_qubits_sweep_list = range(20, 121 * 5 + 1, 50)
else:
    n_total_logical_qubits_sweep_list = range(20, 121 * 5 + 1, 1)
reaction_time_in_cycles_list = [reaction_time_in_cycles]

# We will derive CoreParametersConfig inside the loop for each circuit.
# The total_allowable_rotation_error will also be derived inside the loop.

# The sweep engine expects a physical qubit count and calculates the logical
# qubit budget from it. Since we want to sweep the logical budget directly,
# we must pre-calculate the corresponding physical qubit counts.
phys_qubits_per_logical = 2 * (reference_code_distance + 1) ** 2
n_phys_qubits_total_list = [
    n_log * phys_qubits_per_logical for n_log in n_total_logical_qubits_sweep_list
]

---
### Execution Cell
This cell runs the full sweep. It iterates through each problem size (`n_qubits_data`), pre-computes the specific `CoreParametersConfig` for both HWP and Rz based on their T-counts, and then runs a `run_sweep` for each.

In [ ]:
all_results_rz = []
all_results_hwp = []

for n_data in n_qubits_data_list:
    print(f"\n--- Running sweeps for n_qubits_data = {n_data} ---")

    # 1. Calculate error budgets for this problem size
    total_rot_error = target_rotation_error_per_qubit_to_rotate * n_data
    total_cult_error = target_cultivation_error_per_qubit_to_rotate * n_data

    # 2. Derive circuit-specific configurations for both strategies
    rz_kwargs = frozendict({"n_qubits_data": n_data, "angle": angle})
    rz_core_config, rz_total_rot_error = generate_circuit_specific_configs(
        circuit_builder=build_parallel_rz_for_sweep,
        circuit_builder_kwargs=rz_kwargs,
        total_synthesis_error=total_rot_error,
        total_cultivation_error=total_cult_error,
        phys_error_rate=physical_error_rate,
        reference_code_distance=reference_code_distance,
    )

    hwp_kwargs = frozendict({"n_qubits_data": n_data, "angle": angle})
    hwp_core_config, hwp_total_rot_error = generate_circuit_specific_configs(
        circuit_builder=build_hwp_circuit_for_sweep,
        circuit_builder_kwargs=hwp_kwargs,
        total_synthesis_error=total_rot_error,
        total_cultivation_error=total_cult_error,
        phys_error_rate=physical_error_rate,
        reference_code_distance=reference_code_distance,
    )

    # 3. Run sweeps for this n_data
    print(f"  Running Parallel Rz sweep for n={n_data}...")
    results_rz_n = run_sweep(
        circuit_builder_func=build_parallel_rz_for_sweep,
        circuit_builder_kwargs_list=[rz_kwargs],
        core_configs_list=[rz_core_config],
        total_allowable_rotation_error_list=[rz_total_rot_error],
        reaction_time_in_cycles_list=reaction_time_in_cycles_list,
        flasq_model_configs=flasq_model_configs,
        n_phys_qubits_total_list=n_phys_qubits_total_list,
        print_level=0,
    )
    all_results_rz.extend(results_rz_n)

    print(f"  Running HWP sweep for n={n_data}...")
    results_hwp_n = run_sweep(
        circuit_builder_func=build_hwp_circuit_for_sweep,
        circuit_builder_kwargs_list=[hwp_kwargs],
        core_configs_list=[hwp_core_config],
        total_allowable_rotation_error_list=[hwp_total_rot_error],
        reaction_time_in_cycles_list=reaction_time_in_cycles_list,
        flasq_model_configs=flasq_model_configs,
        n_phys_qubits_total_list=n_phys_qubits_total_list,
        print_level=0,
    )
    all_results_hwp.extend(results_hwp_n)

In [ ]:
# --- Post-Process and Merge DataFrames ---
from pandas import notnull as pdnotnull

# 1. Initial processing
# Note: post_process_for_logical_depth automatically filters out runs where
# Total Logical Qubits < Algorithmic Qubits, correcting the visualization issue.
df_rz_raw = post_process_for_logical_depth(all_results_rz)
df_hwp_raw = post_process_for_logical_depth(all_results_hwp)

# Define pivot columns (common identifiers for a single data point)
pivot_cols = [
    "Number of Logical Qubits",
    "circuit_arg_n_qubits_data",
    "Total Allowable Rotation Error",
    # 'FLASQ Model' will be handled separately
]

# 2. Process Rz Data (Baseline Lines)
# Keep only Full Span for Rz (Zero Span is redundant as Rz has negligible span cost)
df_rz = df_rz_raw[
    df_rz_raw["FLASQ Model"].isin(
        ["Conservative (Full Span)", "Optimistic (Full Span)"]
    )
].copy()
# Clean up the model names (e.g., 'Conservative (Full Span)' -> 'Conservative')
df_rz["FLASQ Model"] = df_rz["FLASQ Model"].str.replace(" (Full Span)", "", regex=False)
df_rz["Rotation Strategy"] = "Parallel Rz"


# 3. Process HWP Data (Bounds)
# Ensure df_hwp_raw is not empty before proceeding
if not df_hwp_raw.empty:
    # Separate Model Base (Conservative/Optimistic) and Span Type (Full/Zero)
    df_hwp_raw["Model Base"] = df_hwp_raw["FLASQ Model"].apply(
        lambda x: "Conservative" if "Conservative" in x else "Optimistic"
    )
    df_hwp_raw["Span Type"] = df_hwp_raw["FLASQ Model"].apply(
        lambda x: "Full Span" if "Full Span" in x else "Zero Span"
    )

    # Pivot the HWP data to get bounds
    # We pivot so that 'Full Span' depth and 'Zero Span' depth are columns.
    df_hwp_bounds = df_hwp_raw.pivot_table(
        index=pivot_cols + ["Model Base"],
        columns="Span Type",
        values="Depth (Logical Timesteps)",
    ).reset_index()

    # Rename columns for clarity
    df_hwp_bounds = df_hwp_bounds.rename(
        columns={
            "Model Base": "FLASQ Model",
            "Zero Span": "Depth Lower Bound (HWP)",
            "Full Span": "Depth Upper Bound (HWP)",
        }
    )
    df_hwp_bounds["Rotation Strategy"] = "HWP"
else:
    # Handle case where HWP failed completely (e.g., insufficient qubits)
    df_hwp_bounds = pd.DataFrame()


# 4. Prepare Final DataFrames for Plotting
# Rename common columns for consistency
rename_dict = {
    "circuit_arg_n_qubits_data": "Number of Qubits to Rotate",
    # "Total Allowable Rotation Error" is now derived, not a primary sweep param
}
df_rz = df_rz.rename(columns=rename_dict)

# Apply renaming to df_hwp_bounds only if it's not empty
if not df_hwp_bounds.empty:
    df_hwp_bounds = df_hwp_bounds.rename(columns=rename_dict)

# df_rz contains the data for the lines.
# df_hwp_bounds contains the data for the shaded regions.

# Filter both dataframes to limit the number of fluid ancillas to a reasonable multiple
# of the number of qubits being rotated. This focuses the plot on more realistic,
# space-constrained scenarios.
if not df_rz.empty:
    df_rz = df_rz[
        df_rz["Number of Logical Qubits"] <= df_rz["Number of Qubits to Rotate"] * 5
    ]

if not df_hwp_bounds.empty:
    # For HWP, fluid_ancillas = total_logical_qubits - 2 * n_qubits_to_rotate
    df_hwp_bounds = df_hwp_bounds[
        df_hwp_bounds["Number of Logical Qubits"]
        <= df_hwp_bounds["Number of Qubits to Rotate"] * 5
    ]

In [ ]:
# --- Plotting ---

import matplotlib.patches as mpatches
from matplotlib.lines import Line2D


def plot_bounded_analysis(
    df_lines, df_bounds, model_name, rot_error_per_qubit, cult_error_per_qubit
):
    """
    Generates the bounded analysis plot for a specific FLASQ model.
    """
    # Filter data for the specific model
    df_l = df_lines[df_lines["FLASQ Model"] == model_name]
    df_b = df_bounds[df_bounds["FLASQ Model"] == model_name]

    if df_l.empty and df_b.empty:
        print(f"No data for model: {model_name}")
        return

    # Setup plot
    plt.figure(figsize=(7.5, 5))
    ax = plt.gca()

    # Determine colors based on 'Number of Qubits to Rotate'
    # Ensure we consider both dataframes in case one strategy is missing data
    n_rotate_l = set(df_l["Number of Qubits to Rotate"].dropna().unique())
    n_rotate_b = set(df_b["Number of Qubits to Rotate"].dropna().unique())
    all_n_rotate = sorted(list(n_rotate_l | n_rotate_b))

    if not all_n_rotate:
        return

    palette = sns.color_palette("tab10", n_colors=len(all_n_rotate))
    color_map = dict(zip(all_n_rotate, palette))

    # Plot Parallel Rz (Solid Lines)
    if not df_l.empty:
        sns.lineplot(
            data=df_l,
            x="Number of Logical Qubits",
            y="Depth (Logical Timesteps)",
            hue="Number of Qubits to Rotate",
            style="Rotation Strategy",  # Ensures solid line by default style map
            palette=color_map,
            ax=ax,
            legend=False,  # We will create a custom legend
            zorder=3,  # Place lines above the shaded regions
        )

    # Plot HWP (Shaded Bounds and Dashed Upper Bound)
    for n_rotate in all_n_rotate:
        df_group = df_b[df_b["Number of Qubits to Rotate"] == n_rotate]
        if df_group.empty:
            continue

        color = color_map[n_rotate]

        # Plot the Upper Bound line (Dashed to represent the heuristic layout)
        sns.lineplot(
            data=df_group,
            x="Number of Logical Qubits",
            y="Depth Upper Bound (HWP)",
            color=color,
            ax=ax,
            legend=False,
            linestyle="--",
            zorder=3,
        )

        # Fill between the bounds (Compilation Range)
        # Ensure data is sorted for fill_between
        df_group = df_group.sort_values(by="Number of Logical Qubits")
        ax.fill_between(
            df_group["Number of Logical Qubits"],
            df_group["Depth Lower Bound (HWP)"],
            df_group["Depth Upper Bound (HWP)"],
            color=color,
            alpha=0.2,
            zorder=2,
        )

    # Aesthetics
    ax.set_yscale("log")
    ax.set_ylabel("Depth (logical timesteps)")
    ax.set_xlabel("Number of logical qubits")
    ax.grid(True, which="both", ls="-", alpha=0.5)
    # Set X-axis limits based on the sweep range
    # ax.set_xlim(left=min(n_total_logical_qubits_sweep_list), right=max(n_total_logical_qubits_sweep_list))
    # ax.set_ylim(bottom=10)

    # Title (Follow Writer's advice)
    title = "Depth Comparison: Parallel Rz vs Hamming Weight Phasing"
    # Omit "Conservative" as it is the default. Add "Optimistic" if applicable.
    if model_name == "Optimistic":
        title += " (Optimistic)"

    # Add subtitle based on the global error budget parameter
    plt.suptitle(title, y=0.97)
    ax.set_title(
        f"Error budget (per qubit to rotate): synthesis = {rot_error_per_qubit:.0e}, cultivation = {cult_error_per_qubit:.0e}"
    )

    # Custom Legend (Crucial Step)
    # We will use two separate legends for clarity.
    # Legend 1: Number of Qubits to Rotate (Colors)
    handles_hue = [
        Line2D([0], [0], color=color_map[n], lw=2, label=str(n)) for n in all_n_rotate
    ]
    legend1 = ax.legend(
        handles=handles_hue,
        title="Number of qubits to rotate",
        loc="upper right",
        bbox_to_anchor=(1.0, 0.75),
    )

    # Legend 2: Rotation Strategy (Styles)
    # Parallel Rz (Solid line)
    handle_rz = Line2D(
        [0], [0], color="black", lw=2, linestyle="-", label="Parallel Rz"
    )
    # HWP Heuristic Layout (Dashed line)
    handle_hwp_upper = Line2D(
        [0], [0], color="black", lw=2, linestyle="--", label="HWP (heuristic layout)"
    )
    # HWP Shaded Region
    handle_hwp_range = mpatches.Patch(
        color="black", alpha=0.2, label="HWP (compilation range)"
    )

    handles_strategy = [handle_rz, handle_hwp_upper, handle_hwp_range]
    legend2 = ax.legend(
        handles=handles_strategy,
        title="Rotation strategy",
        loc="upper right",
        # bbox_to_anchor=(0.7, 0),
    )

    # Add the first legend back to the axes
    ax.add_artist(legend1)

    # plt.tight_layout(rect=[0, 0, 0.85, 0.95])
    plt.tight_layout()
    plt.savefig(
        f"hwp_depth_comparison_{model_name}_roterr_{rot_error_per_qubit:.0e}_culterr_{cult_error_per_qubit:.0e}.pdf",
        bbox_inches="tight",
    )
    plt.show()


# Generate the plots
plot_bounded_analysis(
    df_rz,
    df_hwp_bounds,
    "Conservative",
    target_rotation_error_per_qubit_to_rotate,
    target_cultivation_error_per_qubit_to_rotate,
)
plot_bounded_analysis(
    df_rz,
    df_hwp_bounds,
    "Optimistic",
    target_rotation_error_per_qubit_to_rotate,
    target_cultivation_error_per_qubit_to_rotate,
)

# T-Count Comparison Summary

The T-gate count is a purely logical property of the circuit and does not depend on the physical layout (i.e., the total number of logical qubits). The following table summarizes the T-gate counts for each rotation strategy and shows their ratio.

In [ ]:
# --- Generate and Display T-Count Comparison Table ---
# Combine the raw dataframes for the T-count analysis
# We use the raw dataframes (df_rz_raw, df_hwp_raw) generated in Step 3.
# We need to temporarily add the 'Rotation Strategy' label back in.
df_rz_raw_labeled = df_rz_raw.copy()
df_rz_raw_labeled["Rotation Strategy"] = "Parallel Rz"
df_hwp_raw_labeled = df_hwp_raw.copy()
df_hwp_raw_labeled["Rotation Strategy"] = "HWP"

df_t_count_raw = pd.concat([df_rz_raw_labeled, df_hwp_raw_labeled], ignore_index=True)

# Rename columns for the table
df_t_count_raw = df_t_count_raw.rename(
    columns={
        "circuit_arg_n_qubits_data": "Number of Qubits to Rotate",
        "Total Allowable Rotation Error": "Rotation Synthesis Error",
    }
)


def generate_t_count_comparison_table(df: pd.DataFrame) -> pd.DataFrame:
    """
    Pivots the processed sweep data to create a summary table comparing T-counts.
    """
    # Select only the columns needed for the logical T-count analysis and drop duplicates
    # T-count does not depend on the FLASQ Model or Logical Qubits, so we exclude them.
    df_t_counts = df[
        [
            "Number of Qubits to Rotate",  # "Rotation Synthesis Error" is no longer a primary column
            "Individual Rotation Error",
            "Target Individual T-gate Error",
            "Rotation Strategy",
            "T-Count",
            "T Cultivation Volume",
        ]
    ].drop_duplicates()

    if df_t_counts.empty:
        return pd.DataFrame()

    # Pivot the table to have one column per rotation strategy
    # We pivot only on the number of qubits, as other logical properties (like T-count)
    # are functions of this for a given strategy.
    df_pivot = df_t_counts.pivot_table(
        index=["Number of Qubits to Rotate"],
        columns="Rotation Strategy",
        values="T-Count",
    ).reset_index()

    # Calculate the ratio and rename columns for clarity
    # Ensure columns exist before calculating ratio
    if "Parallel Rz" in df_pivot.columns and "HWP" in df_pivot.columns:
        df_pivot["T-Count Ratio (Rz/HWP)"] = df_pivot["Parallel Rz"] / df_pivot["HWP"]
    else:
        df_pivot["T-Count Ratio (Rz/HWP)"] = np.nan

    df_pivot = df_pivot.rename(
        columns={"Parallel Rz": "Parallel Rz T-Count", "HWP": "HWP T-Count"}
    )

    # Pivot again for the cultivation volume
    df_vol_pivot = df_t_counts.pivot_table(
        index=["Number of Qubits to Rotate"],
        columns="Rotation Strategy",
        values="T Cultivation Volume",
    ).reset_index()
    df_vol_pivot = df_vol_pivot.rename(
        columns={"Parallel Rz": "Rz Cult Vol", "HWP": "HWP Cult Vol"}
    )

    # Pivot for the other columns that vary per strategy
    df_other_pivot = df_t_counts.pivot_table(
        index=["Number of Qubits to Rotate"],
        columns="Rotation Strategy",
        values=["Individual Rotation Error", "Target Individual T-gate Error"],
    ).reset_index()
    # Flatten the multi-level column index
    df_other_pivot.columns = [
        "_".join(col).strip() for col in df_other_pivot.columns.values
    ]

    # Rename the index column back to the original name for merging
    df_other_pivot = df_other_pivot.rename(
        columns={"Number of Qubits to Rotate_": "Number of Qubits to Rotate"}
    )

    # Merge all pivoted dataframes
    df_pivot = pd.merge(df_pivot, df_vol_pivot, on="Number of Qubits to Rotate")
    df_pivot = pd.merge(df_pivot, df_other_pivot, on="Number of Qubits to Rotate")
    return df_pivot


t_count_table = generate_t_count_comparison_table(df_t_count_raw)
print("\n--- T-Count Comparison Summary ---")
# Format numbers nicely
if not t_count_table.empty:
    formatters = {
        "Parallel Rz T-Count": lambda x: f"{x:,.0f}" if pdnotnull(x) else "N/A",
        "HWP T-Count": lambda x: f"{x:,.0f}" if pdnotnull(x) else "N/A",
        "Rz Cult Vol": lambda x: f"{x:,.1f}" if pdnotnull(x) else "N/A",
        "HWP Cult Vol": lambda x: f"{x:,.1f}" if pdnotnull(x) else "N/A",
        "Individual Rotation Error_HWP": lambda x: (
            f"{x:.1e}" if pdnotnull(x) else "N/A"
        ),
        "Individual Rotation Error_Parallel Rz": lambda x: (
            f"{x:.1e}" if pdnotnull(x) else "N/A"
        ),
        "Target Individual T-gate Error_HWP": lambda x: (
            f"{x:.1e}" if pdnotnull(x) else "N/A"
        ),
        "Target Individual T-gate Error_Parallel Rz": lambda x: (
            f"{x:.1e}" if pdnotnull(x) else "N/A"
        ),
        "T-Count Ratio (Rz/HWP)": lambda x: f"{x:.2f}x" if pdnotnull(x) else "N/A",
    }
    # Rename the index column for display
    # The merge key is now "Number of Qubits to Rotate", so rename that column.
    t_count_table = t_count_table.rename(
        columns={"Number of Qubits to Rotate": "Qubits"}
    )
    print(t_count_table.to_string(formatters=formatters))
else:
    print("T-Count table is empty.")

# Computational Volume Comparison Summary

The computational volume is also a logical property of the circuit, but unlike the T-count, it depends on the FLASQ cost model used (Conservative vs. Optimistic). The following table summarizes the computational volumes for each rotation strategy.

In [ ]:
# --- Generate and Display Computational Volume Comparison Table ---


def generate_comp_vol_comparison_table(df: pd.DataFrame) -> pd.DataFrame:
    """
    Pivots the processed sweep data to create a summary table comparing computational volumes.
    """
    # Select only the columns needed for the logical computational volume analysis and drop duplicates.
    # This value does not depend on the number of logical qubits, so we can drop that column.
    df_comp_vols = df[
        [
            "Number of Qubits to Rotate",
            "Individual Rotation Error",
            "Rotation Synthesis Error",
            "Rotation Strategy",
            "FLASQ Model",
            "Total Computational Volume",
            "Raw Measurement Depth",
            "Scaled Measurement Depth",
        ]
    ].drop_duplicates()

    if df_comp_vols.empty:
        return pd.DataFrame()

    # Pivot the table to have one column per rotation strategy
    df_pivot = df_comp_vols.pivot_table(
        index=[
            "Number of Qubits to Rotate",
            "FLASQ Model",
        ],
        columns="Rotation Strategy",
        values="Total Computational Volume",
    ).reset_index()

    # Calculate the ratio of computational volumes
    if "Parallel Rz" in df_pivot.columns and "HWP" in df_pivot.columns:
        df_pivot["Comp Vol Ratio (Rz/HWP)"] = df_pivot["Parallel Rz"] / df_pivot["HWP"]
    else:
        df_pivot["Comp Vol Ratio (Rz/HWP)"] = np.nan

    # Pivot for Raw Measurement Depth
    df_raw_depth_pivot = df_comp_vols.pivot_table(
        index=["Number of Qubits to Rotate", "FLASQ Model"],
        columns="Rotation Strategy",
        values="Raw Measurement Depth",
    ).reset_index()
    df_pivot = pd.merge(
        df_pivot,
        df_raw_depth_pivot,
        on=["Number of Qubits to Rotate", "FLASQ Model"],
        suffixes=("", "_RawDepth"),
    )

    # Pivot for Scaled Measurement Depth
    df_scaled_depth_pivot = df_comp_vols.pivot_table(
        index=["Number of Qubits to Rotate", "FLASQ Model"],
        columns="Rotation Strategy",
        values="Scaled Measurement Depth",
    ).reset_index()
    df_pivot = pd.merge(
        df_pivot,
        df_scaled_depth_pivot,
        on=["Number of Qubits to Rotate", "FLASQ Model"],
        suffixes=("", "_ScaledDepth"),
    )

    return df_pivot


comp_vol_table = generate_comp_vol_comparison_table(df_t_count_raw)
print("\n--- Computational Volume Comparison Summary ---")
if not comp_vol_table.empty:
    formatters = {
        "Parallel Rz": lambda x: f"{x:,.0f}" if pdnotnull(x) else "N/A",
        "HWP": lambda x: f"{x:,.0f}" if pdnotnull(x) else "N/A",
        "Comp Vol Ratio (Rz/HWP)": lambda x: f"{x:.2f}x" if pdnotnull(x) else "N/A",
        "HWP_RawDepth": lambda x: f"{x:,.1f}" if pdnotnull(x) else "N/A",
        "Parallel Rz_RawDepth": lambda x: f"{x:,.1f}" if pdnotnull(x) else "N/A",
        "HWP_ScaledDepth": lambda x: f"{x:,.1f}" if pdnotnull(x) else "N/A",
        "Parallel Rz_ScaledDepth": lambda x: f"{x:,.1f}" if pdnotnull(x) else "N/A",
    }
    print(comp_vol_table.to_string(formatters=formatters))
else:
    print("Computational Volume table is empty.")

# Analysis with a Different Error Budget

To explore the sensitivity of our results to the error budget, we repeat the analysis for the Conservative model but with a less stringent rotation synthesis error target of `1e-5` per qubit.

In [ ]:
# --- Define Parameters for the New Sweep ---
target_rotation_error_per_qubit_to_rotate_new = 1e-5

# We only need the conservative models for this plot.
flasq_model_configs_new_sweep = [
    (conservative_FLASQ_costs, "Conservative (Full Span)"),
    (conservative_zero_span, "Conservative (Zero Span)"),
]

# --- Execute the New Sweep ---
all_results_rz_new = []
all_results_hwp_new = []

for n_data in n_qubits_data_list:
    print(f"\n--- Running new sweep for n_qubits_data = {n_data} ---")

    # 1. Calculate error budgets using the new target
    total_rot_error_new = target_rotation_error_per_qubit_to_rotate_new * n_data
    # Keep cultivation error the same for a controlled comparison
    total_cult_error = target_cultivation_error_per_qubit_to_rotate * n_data

    # 2. Derive circuit-specific configurations
    rz_kwargs = frozendict({"n_qubits_data": n_data, "angle": angle})
    rz_core_config, rz_total_rot_error = generate_circuit_specific_configs(
        circuit_builder=build_parallel_rz_for_sweep,
        circuit_builder_kwargs=rz_kwargs,
        total_synthesis_error=total_rot_error_new,
        total_cultivation_error=total_cult_error,
        phys_error_rate=physical_error_rate,
        reference_code_distance=reference_code_distance,
    )

    hwp_kwargs = frozendict({"n_qubits_data": n_data, "angle": angle})
    hwp_core_config, hwp_total_rot_error = generate_circuit_specific_configs(
        circuit_builder=build_hwp_circuit_for_sweep,
        circuit_builder_kwargs=hwp_kwargs,
        total_synthesis_error=total_rot_error_new,
        total_cultivation_error=total_cult_error,
        phys_error_rate=physical_error_rate,
        reference_code_distance=reference_code_distance,
    )

    # 3. Run sweeps
    results_rz_n = run_sweep(
        circuit_builder_func=build_parallel_rz_for_sweep,
        circuit_builder_kwargs_list=[rz_kwargs],
        core_configs_list=[rz_core_config],
        total_allowable_rotation_error_list=[rz_total_rot_error],
        reaction_time_in_cycles_list=reaction_time_in_cycles_list,
        flasq_model_configs=flasq_model_configs_new_sweep,
        n_phys_qubits_total_list=n_phys_qubits_total_list,
        print_level=0,
    )
    all_results_rz_new.extend(results_rz_n)

    results_hwp_n = run_sweep(
        circuit_builder_func=build_hwp_circuit_for_sweep,
        circuit_builder_kwargs_list=[hwp_kwargs],
        core_configs_list=[hwp_core_config],
        total_allowable_rotation_error_list=[hwp_total_rot_error],
        reaction_time_in_cycles_list=reaction_time_in_cycles_list,
        flasq_model_configs=flasq_model_configs_new_sweep,
        n_phys_qubits_total_list=n_phys_qubits_total_list,
        print_level=0,
    )
    all_results_hwp_new.extend(results_hwp_n)

# --- Post-Process and Plot for the New Sweep ---
df_rz_raw_new = post_process_for_logical_depth(all_results_rz_new)
df_hwp_raw_new = post_process_for_logical_depth(all_results_hwp_new)

# Process Rz Data
df_rz_new = df_rz_raw_new.copy()
df_rz_new["FLASQ Model"] = "Conservative"
df_rz_new["Rotation Strategy"] = "Parallel Rz"

# Process HWP Data
if not df_hwp_raw_new.empty:
    df_hwp_raw_new["Model Base"] = "Conservative"
    df_hwp_raw_new["Span Type"] = df_hwp_raw_new["FLASQ Model"].apply(
        lambda x: "Full Span" if "Full Span" in x else "Zero Span"
    )
    df_hwp_bounds_new = df_hwp_raw_new.pivot_table(
        index=["circuit_arg_n_qubits_data", "Number of Logical Qubits", "Model Base"],
        columns="Span Type",
        values="Depth (Logical Timesteps)",
    ).reset_index()
    df_hwp_bounds_new = df_hwp_bounds_new.rename(
        columns={
            "Model Base": "FLASQ Model",
            "Zero Span": "Depth Lower Bound (HWP)",
            "Full Span": "Depth Upper Bound (HWP)",
        }
    )
    df_hwp_bounds_new["Rotation Strategy"] = "HWP"
else:
    df_hwp_bounds_new = pd.DataFrame()

# Rename columns for plotting
df_rz_new = df_rz_new.rename(columns=rename_dict)
if not df_hwp_bounds_new.empty:
    df_hwp_bounds_new = df_hwp_bounds_new.rename(columns=rename_dict)

# Filter the new dataframes to the same reasonable multiple
if not df_rz_new.empty:
    df_rz_new = df_rz_new[
        df_rz_new["Number of Logical Qubits"]
        <= df_rz_new["Number of Qubits to Rotate"] * 5
    ]
if not df_hwp_bounds_new.empty:
    df_hwp_bounds_new = df_hwp_bounds_new[
        df_hwp_bounds_new["Number of Logical Qubits"]
        <= df_hwp_bounds_new["Number of Qubits to Rotate"] * 5
    ]

# Generate the plot
plot_bounded_analysis(
    df_rz_new,
    df_hwp_bounds_new,
    "Conservative",
    target_rotation_error_per_qubit_to_rotate_new,
    target_cultivation_error_per_qubit_to_rotate,
)